# Evaluación Sumativa 3: Modelamiento Predictivo Integrado

**Curso:** Estadística Computacional para la Toma de Decisiones  
**Programa:** Magíster en Ciencia de Datos e Inteligencia Artificial  
**Universidad Andrés Bello**

---

## Información del Equipo

| Campo | Información |
|-------|-------------|
| **Número de Grupo** | 4 |
| **Integrante 1** | Jeremmy Díaz Alfaro |
| **Integrante 2** | Karim Zaid Sánchez |
| **Integrante 3** | Laura Arenas Kosin |
| **Dataset Seleccionado** | Rain in Australia (weatherAUS, Kaggle) |
| **Fecha de Entrega** | 27/10/2026 |

## Integración con S1 y S2

Esta evaluación integra todo el proyecto:

**De S1:** análisis de faltantes, matriz de correlación, outliers identificados (`01_formativa1_eda.ipynb`).

**De S2:** correlaciones estables y bootstrap para IC de coeficientes y OR (`02_sumativa2_validacion.ipynb`, `data/processed/resultados_validados_s2.csv`).

Este notebook amplía la Formativa 2 en cuatro puntos: (1) imputación por regresión lineal múltiple y comparación de tres estrategias de faltantes, (2) tres modelos logísticos con criterios de selección distintos, (3) bootstrap de los coeficientes del mejor modelo y (4) ponderación de clase para mejorar el recall. Todos los cálculos usan la semilla `42`.

## Configuración del Entorno

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix)

warnings.filterwarnings("ignore", category=FutureWarning)

# buscamos la carpeta del proyecto para poder importar src
raiz = Path.cwd().resolve()
while not (raiz / "src").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.append(str(raiz))
from src.carga_datos import cargar_weather

SEED = 42
FIGURAS = raiz / "figuras"
FIGURAS.mkdir(exist_ok=True)
PROCESADOS = raiz / "data" / "processed"
PROCESADOS.mkdir(parents=True, exist_ok=True)

# misma paleta de la S1 y la S2
INK = '#2b2b2b'; TEAL = '#2e6b75'; TERRA = '#a35140'; OCHRE = '#c4862f'; STEM = '#bdb3a3'
BG = '#f6f1ea'; GRID = '#ddd5c6'
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.family': 'serif', 'font.size': 10,
    'axes.facecolor': BG, 'axes.edgecolor': '#8c8578',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 12, 'axes.titlelocation': 'left',
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
})
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

TARGET = 'RainTomorrow'

def guardar(fig, nombre):
    fig.savefig(FIGURAS / nombre, bbox_inches='tight', facecolor='white')

# misma limpieza básica de la S1: la nubosidad 9 no es válida en octas
df = cargar_weather()
for c in ['Cloud9am', 'Cloud3pm']:
    df.loc[df[c] > 8, c] = np.nan
df['RainToday'] = df['RainToday'].map({'Yes': 1, 'No': 0})   # codificación binaria

# el objetivo nunca se imputa: se descartan las filas sin RainTomorrow
datos = df.dropna(subset=[TARGET]).reset_index(drop=True)
datos['y'] = (datos[TARGET] == 'Yes').astype(int)

VARS_NUM = ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed',
            'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am',
            'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday']
HEAVY = ['Sunshine', 'Evaporation', 'Cloud3pm', 'Cloud9am']        # variables con 38-48 % de faltantes
MENORES = [v for v in VARS_NUM if v not in HEAVY]                  # faltantes bajos (< 11 %)

print(f"Filas totales: {len(df):,} | con objetivo: {len(datos):,} | descartadas por objetivo vacío: {len(df) - len(datos):,}")
print(f"Proporción de lluvia mañana (filas válidas): {datos['y'].mean():.4f}")

---
# 0. Carga de Resultados de S1 y S2

### Resultados de la S1 que se usan en esta evaluación

Se documentan con las mismas definiciones de la S1 (filas con `RainTomorrow`, nubosidad válida en octas): porcentaje de faltantes por variable, matriz de correlaciones con pares completos y los valores extremos identificados.

In [ ]:
# resultados de S1
faltantes_s1 = (datos[VARS_NUM].isna().mean() * 100).sort_values(ascending=False)       # % de faltantes por variable
correlaciones_s1 = datos[VARS_NUM + ['y']].corr()                                       # matriz de correlaciones (pares completos)
outliers_s1 = pd.DataFrame({'Asimetría': datos[VARS_NUM].skew(), 'Percentil 99': datos[VARS_NUM].quantile(0.99),
                            'Máximo': datos[VARS_NUM].max()}).sort_values('Asimetría', ascending=False)

print("Faltantes de S1 (% por variable, las 5 más altas):"); display(faltantes_s1.head(5).round(1).to_frame('% faltante').T)
print("Correlaciones de S1 con la variable objetivo (y = llueve mañana):")
display(correlaciones_s1['y'].drop('y').sort_values(key=abs, ascending=False).round(3).to_frame('r con y').T)
print("Valores extremos de S1 (variables más asimétricas):"); display(outliers_s1.head(4).round(2))

### Resultados de la S2 que se usan en esta evaluación

Valores tomados del informe de la S2 (Tabla 1 y sección 3) y del archivo `resultados_validados_s2.csv`.

In [ ]:
# resultados de S2 (informe S2: IC bootstrap al 95 % de correlaciones y parámetros)
correlaciones_estables_s2 = [
    # (par de variables, r, IC bootstrap 95 %, comentario)
    ('Humidity3pm - RainTomorrow', 0.446, (0.442, 0.451), 'robusta; la más fuerte con el objetivo'),
    ('Sunshine - Cloud3pm', -0.704, (-0.708, -0.701), 'robusta; dos variables muy útiles pero con 40-48 % de faltantes'),
    ('Pressure9am - Pressure3pm', 0.961, (0.961, 0.962), 'casi repetidas: no usar juntas'),
    ('WindGustSpeed - WindSpeed3pm', 0.686, (0.683, 0.690), 'casi repetidas: no usar juntas'),
    ('Temp9am - RainTomorrow', -0.026, (-0.031, -0.021), 'estable pero casi nula'),
]
parametros_validados_s2 = {
    'MaxTemp (media)':        {'estimacion': 23.22, 'ic_bca': (23.185, 23.258)},
    'Rainfall (media, mm)':   {'estimacion': 2.361, 'ic_bca': (2.317, 2.405), 'nota': 'muy sensible a días extremos'},
    'Humidity3pm (media)':    {'estimacion': 51.54, 'ic_bca': (51.431, 51.645)},
    'Proporción lluvia mañana': {'estimacion': 0.2242, 'ic_bca': (0.2220, 0.2263), 'ic_por_ciudad': (0.2055, 0.2420)},
}
validados_s2 = pd.read_csv(PROCESADOS / 'resultados_validados_s2.csv')
print("Correlaciones estables de S2:")
display(pd.DataFrame(correlaciones_estables_s2, columns=['Par', 'r', 'IC 95 %', 'Comentario']).set_index('Par'))
print("Resultados de S1 validados en S2 (resultados_validados_s2.csv):"); display(validados_s2.set_index('Resultado'))

# PARTE 1: Manejo de Datos Faltantes

## 1.1 Resumen de Análisis de S1

En la S1 (Figura 1) se vio que 21 de las 23 variables tienen faltantes, y en la S2 (Tabla 3) se marcó que el ~40 % de faltantes de la nubosidad **no es al azar** porque hay estaciones que no miden ciertas variables. Aquí no repetimos el análisis: resumimos lo que necesitamos para decidir cómo imputar.

In [ ]:
faltantes = pd.DataFrame({
    'n faltantes': datos[VARS_NUM].isna().sum(),
    '% faltante': datos[VARS_NUM].isna().mean() * 100,
}).sort_values('% faltante', ascending=False)

por_ciudad = datos.groupby('Location')[HEAVY].apply(lambda g: g.isna().mean() * 100)
filas = []
for h in HEAVY:
    sin = por_ciudad.index[por_ciudad[h] == 100]
    en_sin = datos.loc[datos['Location'].isin(sin), h].isna().sum()
    filas.append([h, datos[h].isna().sum(), datos[h].isna().mean() * 100, len(sin), en_sin / datos[h].isna().sum() * 100])
estructural = pd.DataFrame(filas, columns=['Variable', 'n faltantes', '% faltante',
                                           'Ciudades que nunca la miden (de 49)',
                                           '% de los faltantes en esas ciudades']).set_index('Variable')

n_completos = datos[VARS_NUM].dropna().shape[0]
print(f"Filas con las 17 variables completas: {n_completos:,} ({n_completos / len(datos) * 100:.1f} %), "
      f"en {datos.dropna(subset=VARS_NUM)['Location'].nunique()} de 49 ciudades\n")
print("Tabla 1. Faltantes por variable (filas con objetivo)")
display(faltantes.round(2))
print("Tabla 2. Faltantes estructurales de las cuatro variables a imputar")
display(estructural.round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 9))
orden = por_ciudad.sort_values('Sunshine').index
im = ax.imshow(por_ciudad.loc[orden], aspect='auto', cmap='YlOrBr', vmin=0, vmax=100)
ax.set_xticks(range(len(HEAVY))); ax.set_xticklabels(HEAVY)
ax.set_yticks(range(len(orden))); ax.set_yticklabels(orden, fontsize=7)
ax.grid(False)
ax.set_title('Figura 1 - % de faltantes por ciudad y variable')
fig.colorbar(im, ax=ax, shrink=0.5, label='% faltante')
fig.tight_layout(); guardar(fig, 's3_fig_faltantes_ciudad.png'); plt.show()

**Lectura.** Las cuatro variables críticas tienen entre 37,7 % (Cloud9am) y 47,7 % (Sunshine) de faltantes entre las filas con objetivo (en la S1 se reportó 48 % y 41 % para Sunshine y Cloud3pm, calculado sobre las 145.460 filas). Entre 12 y 19 de las 49 ciudades **nunca** miden cada variable, y esas ciudades concentran entre 60 % y 77 % de los faltantes (Tabla 2 y Figura 1). Si se eliminaran las filas con algún faltante quedarían 58.089 (40,9 %), de solo 26 ciudades.

Esto confirma lo que ya había advertido la S2: el faltante no es al azar, es **estructural** (depende de la estación). Por eso el supuesto MAR es aproximado y se trata como limitación.

## 1.2 Imputación por Regresión Lineal

**Supuesto sobre el mecanismo.** Para imputar por regresión suponemos que los faltantes son **MAR** (dependen de variables que sí observamos, como la humedad o la temperatura) y no de su propio valor. La Figura 1 muestra que el supuesto es parcial: una parte grande de los faltantes es **estructural**, porque la estación nunca mide la variable. Eso es más cercano a MNAR y lo declaramos como **limitación**: en esas ciudades el valor imputado es una extrapolación a partir de otras estaciones. Más abajo medimos cuánto pesa esa extrapolación.

**Procedimiento.**
1. Las 13 variables con pocos faltantes (< 11 %) se rellenan primero con su mediana, igual en las tres estrategias, para que los predictores estén completos. Así las estrategias solo difieren en cómo tratan las cuatro variables críticas.
2. Para cada variable crítica se eligen **3 a 5 predictores** con la matriz de correlaciones de la S1 (calculada con pares completos): se ordenan por |r| con un bono de 0,05 para las variables que S2 validó como robustas (humedad 3 pm, lluvia de hoy, presión), se exige |r| ≥ 0,10 y se descarta cualquier predictor casi repetido (|r| ≥ 0,85) con uno ya elegido.
3. Se ajusta una regresión lineal **solo con las observaciones completas**, se revisan los residuos y se valida el desempeño.
4. Se imputan los vacíos, acotando a los valores válidos (y redondeando las octas de nubosidad).
5. **`RainTomorrow` no se usa como predictor ni se imputa.**

In [ ]:
# paso 1: faltantes bajos con su mediana (comunes a todas las estrategias)
medianas = datos[MENORES].median()
base = datos.copy()
base[MENORES] = base[MENORES].fillna(medianas)

# paso 2: matriz de correlaciones de la S1 (pares completos) y selección de predictores
corr_s1 = correlaciones_s1   # matriz de correlaciones de la S1 (sección 0)
S2_VALIDADAS = {'Humidity3pm', 'RainToday', 'Pressure9am', 'Pressure3pm'}

def elegir_predictores(obj, candidatos, corr, max_k=5, r_min=0.10, r_max_entre=0.85):
    orden = sorted(candidatos, key=lambda v: abs(corr.loc[obj, v]) + (0.05 if v in S2_VALIDADAS else 0), reverse=True)
    elegidos = []
    for v in orden:
        if abs(corr.loc[obj, v]) < r_min:
            continue
        if all(abs(corr.loc[v, e]) < r_max_entre for e in elegidos):
            elegidos.append(v)
        if len(elegidos) == max_k:
            break
    assert len(elegidos) >= 3, f'{obj}: menos de 3 predictores'
    return elegidos

PREDICTORES = {h: elegir_predictores(h, MENORES, corr_s1) for h in HEAVY}
sel = pd.DataFrame([[h, ', '.join(f"{p} ({corr_s1.loc[h, p]:+.2f})" for p in ps)] for h, ps in PREDICTORES.items()],
                   columns=['Variable a imputar', 'Predictores elegidos (r de S1)']).set_index('Variable a imputar')
print("Tabla 3. Predictores elegidos con la matriz de correlaciones de la S1")
display(sel)

In [ ]:
def imputar_por_regresion(df, var_objetivo, predictores, limites=None, redondear=False):
    """Imputa los faltantes de `var_objetivo` con una regresión lineal múltiple.

    El modelo se ajusta solo con las observaciones completas. Los predictores deben venir sin faltantes
    y `RainTomorrow` nunca se usa. Devuelve (valores imputados para las filas faltantes, modelo OLS ajustado).
    `limites` acota el valor al rango observado y `redondear` redondea (octas de nubosidad).
    """
    obs = df[var_objetivo].notna()
    modelo = sm.OLS(df.loc[obs, var_objetivo], sm.add_constant(df.loc[obs, predictores])).fit()
    X_faltan = sm.add_constant(df.loc[~obs, predictores], has_constant='add')
    imputado = modelo.predict(X_faltan)
    if limites is not None:
        imputado = imputado.clip(*limites)
    if redondear:
        imputado = imputado.round()
    return imputado, modelo

In [ ]:
# paso 3: ajuste con observaciones completas e imputación (acotada al rango observado; octas redondeadas)
LIMITES = {h: (datos[h].min(), datos[h].max()) for h in HEAVY}
modelos_imp, imputados, filas_coef = {}, {}, []
for h, ps in PREDICTORES.items():
    imputados[h], modelos_imp[h] = imputar_por_regresion(base, h, ps, LIMITES[h], redondear=h.startswith('Cloud'))
    m = modelos_imp[h]
    for nom in m.params.index:
        filas_coef.append([h, nom, m.params[nom], m.bse[nom], m.tvalues[nom]])
coef_imp = pd.DataFrame(filas_coef, columns=['Variable imputada', 'Término', 'Coef.', 'Error est.', 't']).set_index(['Variable imputada', 'Término'])
print("Tabla 4. Coeficientes de las cuatro regresiones de imputación (observaciones completas)")
display(coef_imp)

# paso 4: se rellenan los vacíos con los valores predichos
base_reg = base.copy()
for h in HEAVY:
    base_reg.loc[datos[h].isna(), h] = imputados[h]
print("Valores imputados por regresión:", {h: int(datos[h].isna().sum()) for h in HEAVY})
print("Total de celdas imputadas por regresión:", int(datos[HEAVY].isna().sum().sum()))

**Validación del desempeño.** Un R² calculado sobre los mismos datos con que se ajustó el modelo es optimista. Por eso evaluamos de dos maneras con observaciones que el modelo no vio:

- **Aleatoria 80/20:** simula faltantes dispersos.
- **Dejando ciudades fuera (GroupKFold, 5 grupos):** simula lo que realmente pasa con los faltantes estructurales, porque predice en ciudades que el modelo nunca vio.

El RMSE se compara con el de la imputación por mediana.

In [ ]:
def validar(h, ps):
    obs = np.flatnonzero(datos[h].notna().to_numpy())
    X = base.iloc[obs][ps].to_numpy(); yv = datos.iloc[obs][h].to_numpy(); g = datos.iloc[obs]['Location'].to_numpy()

    def evalua(i_tr, i_te):
        A = np.column_stack([np.ones(len(i_tr)), X[i_tr]])
        b = np.linalg.lstsq(A, yv[i_tr], rcond=None)[0]
        pred = np.column_stack([np.ones(len(i_te)), X[i_te]]) @ b
        e = yv[i_te] - pred
        rmse = np.sqrt(np.mean(e ** 2))
        rmse0 = np.sqrt(np.mean((yv[i_te] - np.median(yv[i_tr])) ** 2))
        r2 = 1 - np.sum(e ** 2) / np.sum((yv[i_te] - yv[i_te].mean()) ** 2)
        return rmse, rmse0, r2

    a, b = train_test_split(np.arange(len(obs)), test_size=0.2, random_state=SEED)
    al = evalua(a, b)
    ci = np.mean([evalua(tr_, te_) for tr_, te_ in GroupKFold(5).split(X, yv, g)], axis=0)
    return al, ci

filas = []
for h, ps in PREDICTORES.items():
    al, ci = validar(h, ps)
    n_imp = int(datos[h].isna().sum())
    sin = por_ciudad.index[por_ciudad[h] == 100]
    n_sin = int(datos.loc[datos['Location'].isin(sin), h].isna().sum())
    filas.append([h, len(ps), int(datos[h].notna().sum()), n_imp, n_imp / len(datos) * 100,
                  modelos_imp[h].rsquared, al[2], ci[2], al[0], ci[0], ci[1], n_sin / n_imp * 100])
tabla_imp = pd.DataFrame(filas, columns=['Variable', 'n pred.', 'n completos', 'n imputados', '% imputado',
                                         'R² (ajuste)', 'R² (80/20)', 'R² (ciudades fuera)',
                                         'RMSE (80/20)', 'RMSE (ciudades fuera)', 'RMSE mediana (ciudades fuera)',
                                         '% imputados en ciudades sin la medición']).set_index('Variable')
print("Tabla 5. Desempeño de la imputación por regresión")
display(tabla_imp.round(3))

In [ ]:
# supuestos de la regresión: residuos vs. ajustados y normalidad (muestra de 4.000 puntos)
rng = np.random.default_rng(SEED)
fig, ejes = plt.subplots(2, 4, figsize=(14, 6.5))
for j, h in enumerate(HEAVY):
    m = modelos_imp[h]
    k = rng.choice(len(m.resid), size=4000, replace=False)
    aj = m.fittedvalues.to_numpy()[k]; rs = m.resid.to_numpy()[k]
    ejes[0, j].scatter(aj, rs, s=4, alpha=0.35, color=TEAL)
    ejes[0, j].axhline(0, color=TERRA, lw=1)
    ejes[0, j].set_title(h); ejes[0, j].set_xlabel('Valor ajustado'); ejes[0, j].set_ylabel('Residuo')
    stats.probplot(m.resid.to_numpy()[k], dist='norm', plot=ejes[1, j])
    ejes[1, j].get_lines()[0].set(markersize=2, color=TEAL); ejes[1, j].get_lines()[1].set(color=TERRA)
    ejes[1, j].set_title(''); ejes[1, j].set_xlabel('Cuantiles normales'); ejes[1, j].set_ylabel('Cuantiles de residuos')
fig.suptitle('Figura 2 - Residuos de las regresiones de imputación (arriba: homocedasticidad; abajo: normalidad)', x=0.01, ha='left')
fig.tight_layout(); guardar(fig, 's3_fig_residuos_imputacion.png'); plt.show()

**Lectura.** Se imputan 239.413 celdas: Sunshine 67.816, Evaporation 60.843, Cloud3pm 57.095 y Cloud9am 53.659.

- **Desempeño.** Las regresiones explican entre 29 % (Cloud9am) y 44 % (Sunshine) de la variabilidad. Ese R² casi no baja con datos que el modelo no vio: 0,29 a 0,44 en la validación 80/20 y 0,28 a 0,43 dejando ciudades completas fuera. En RMSE, la regresión supera a la mediana en las cuatro variables al dejar ciudades fuera (por ejemplo Sunshine 2,85 vs. 3,89 horas y Cloud3pm 2,29 vs. 2,76 octas).
- **Supuestos.** Los residuos no son perfectamente normales: Evaporation tiene una cola derecha larga (hay un día de 145 mm) y las nubosidades, que son octas enteras, forman las diagonales de la figura. Eso no cambia el valor predicho, pero sí significa que la incertidumbre de lo imputado no es la de un modelo normal.
- **Límite de la validación.** La validación por ciudades deja fuera ciudades que sí miden la variable. **No hay forma de comprobar** el desempeño en las 12 a 19 ciudades que nunca la miden, y ahí cae entre 60 % y 77 % de lo imputado (última columna de la Tabla 5). Esa es la limitación del supuesto MAR que declaramos.

## 1.3 Comparación de Estrategias

Comparamos las tres estrategias en lo que pide la plantilla: **tamaño muestral**, **distribución de las variables** y **preservación de las correlaciones de la S1**. La referencia de la S1 son las medias, desviaciones y correlaciones calculadas con los datos observados (pares completos).

Las tres versiones del dataset: **A** = eliminación de casos con algún faltante, **B** = imputación simple por mediana (la de la Formativa 2) y **C** = imputación por regresión.

In [ ]:
# las tres versiones del dataset
ds_elim = datos.dropna(subset=VARS_NUM).copy()           # A: eliminación de casos

ds_med = base.copy()                                     # B: imputación simple (mediana)
for h in HEAVY:
    ds_med[h] = datos[h].fillna(datos[h].median())

ds_reg = base_reg.copy()                                 # C: imputación por regresión

assert ds_reg[VARS_NUM].isna().sum().sum() == 0 and ds_med[VARS_NUM].isna().sum().sum() == 0
assert (ds_reg['y'] == datos['y']).all()                 # el objetivo no se tocó

COLS_SALIDA = ['Date', 'Location', TARGET, 'y'] + VARS_NUM
for nombre, ds in [('eliminacion', ds_elim), ('mediana', ds_med), ('regresion', ds_reg)]:
    ds[COLS_SALIDA].to_csv(PROCESADOS / f'weather_{nombre}.csv.gz', index=True, index_label='id', compression={'method': 'gzip', 'compresslevel': 1})
print("Guardados en data/processed/: weather_eliminacion.csv.gz, weather_mediana.csv.gz, weather_regresion.csv.gz")

In [ ]:
filas = []
for nombre, ds in [('Eliminación', ds_elim), ('Mediana', ds_med), ('Regresión', ds_reg)]:
    filas.append([nombre, len(ds), len(ds) / len(datos) * 100, ds['Location'].nunique(), ds['y'].mean()])
tam = pd.DataFrame(filas, columns=['Estrategia', 'n', '% de las filas con objetivo', 'Ciudades', 'Prop. lluvia mañana']).set_index('Estrategia')
print("Tabla 6. Tamaño muestral por estrategia")
display(tam.round(3))

filas = []
for h in HEAVY:
    ref = datos[h].dropna()
    for nombre, s in [('Observados (S1)', ref), ('Eliminación', ds_elim[h]), ('Mediana', ds_med[h]), ('Regresión', ds_reg[h])]:
        filas.append([h, nombre, len(s), s.mean(), s.std(), s.median()])
dist = pd.DataFrame(filas, columns=['Variable', 'Estrategia', 'n', 'Media', 'Desv. est.', 'Mediana']).set_index(['Variable', 'Estrategia'])
print("Tabla 7. Distribución de las variables críticas")
display(dist.round(3))

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, h in zip(ejes, HEAVY):
    bins = np.linspace(LIMITES[h][0], LIMITES[h][1], 40)
    ax.hist(datos[h].dropna(), bins=bins, density=True, histtype='stepfilled', color=STEM, alpha=0.6, label='Observados')
    ax.hist(ds_med[h], bins=bins, density=True, histtype='step', color=OCHRE, lw=1.6, label='Mediana')
    ax.hist(ds_reg[h], bins=bins, density=True, histtype='step', color=TEAL, lw=1.6, label='Regresión')
    ax.set_title(h); ax.set_yticks([])
ejes[0].legend(frameon=False, fontsize=8)
fig.suptitle('Figura 3 - Distribución de las cuatro variables según la estrategia de imputación', x=0.01, ha='left')
fig.tight_layout(); guardar(fig, 's3_fig_imputacion_distribuciones.png'); plt.show()

In [ ]:
# preservación de correlaciones: se compara cada matriz con la de la S1 (pares completos)
ref_corr = datos[VARS_NUM + ['y']].corr()
pares_heavy = [(a, b) for a in HEAVY for b in VARS_NUM + ['y'] if a != b]
filas = []
for nombre, ds in [('Eliminación', ds_elim), ('Mediana', ds_med), ('Regresión', ds_reg)]:
    c = ds[VARS_NUM + ['y']].corr()
    dif = (c - ref_corr).abs()
    dif_h = np.mean([dif.loc[a, b] for a, b in pares_heavy])
    filas.append([nombre, dif.where(~np.eye(len(dif), dtype=bool)).stack().mean(), dif_h,
                  *[c.loc[h, 'y'] for h in HEAVY]])
corr_tab = pd.DataFrame(filas, columns=['Estrategia', '|Δr| medio (todas)', '|Δr| medio (con las 4 críticas)'] +
                        [f'r({h}, y)' for h in HEAVY]).set_index('Estrategia')
corr_tab.loc['S1 (referencia)'] = [0, 0] + [ref_corr.loc[h, 'y'] for h in HEAVY]
print("Tabla 8. Correlaciones: diferencia con la matriz de la S1 y correlación de cada variable crítica con el objetivo")
display(corr_tab.round(3))

# correlaciones entre las propias variables críticas
pares = [('Sunshine', 'Cloud3pm'), ('Sunshine', 'Humidity3pm'), ('Cloud3pm', 'Humidity3pm'), ('Evaporation', 'MaxTemp'), ('Cloud9am', 'Cloud3pm')]
filas = [[f'{a} - {b}', ref_corr.loc[a, b], *[ds[[a, b]].corr().iloc[0, 1] for ds in (ds_elim, ds_med, ds_reg)]] for a, b in pares]
print("Tabla 9. Correlaciones clave entre pares de variables")
display(pd.DataFrame(filas, columns=['Par', 'S1 (observados)', 'Eliminación', 'Mediana', 'Regresión']).set_index('Par').round(3))

In [ ]:
# criterio objetivo para justificar la estrategia: conserva n y es la que mejor preserva las correlaciones de la S1
cand = {'Mediana': corr_tab.loc['Mediana', '|Δr| medio (con las 4 críticas)'],
        'Regresión': corr_tab.loc['Regresión', '|Δr| medio (con las 4 críticas)']}
mejor = min(cand, key=cand.get)
print("Estrategia con menor distorsión de correlaciones entre las que conservan todos los casos:", mejor)
print(f"Eliminación conserva solo {len(ds_elim) / len(datos) * 100:.1f} % de las filas y {ds_elim['Location'].nunique()} de 49 ciudades.")

**Lectura y decisión.**

- **A. Eliminación:** conserva 58.089 filas (40,9 %) y 26 de las 49 ciudades. Sus correlaciones se parecen a las de la S1 (|Δr| medio de 0,016 con las cuatro variables críticas), pero porque describe otra población: las ciudades que miden todo. Un modelo así no serviría para las otras 23.
- **B. Mediana:** conserva los 142.193 casos pero aplana las variables. La desviación estándar de Sunshine baja de 3,78 a 2,77 y la de Cloud3pm de 2,72 a 2,12 (Tabla 7), y las correlaciones se debilitan: Sunshine–Cloud3pm pasa de −0,704 a −0,528 y la correlación de Sunshine con la lluvia de mañana, de −0,451 a −0,324. |Δr| medio: 0,067.
- **C. Regresión:** conserva los 142.193 casos y las 49 ciudades, recupera mejor la dispersión (Sunshine 3,27 y Cloud3pm 2,30) y la correlación con el objetivo: Sunshine −0,429 (S1: −0,451), Evaporation −0,141 (−0,119), Cloud3pm 0,386 (0,382) y Cloud9am 0,329 (0,317). |Δr| medio: 0,042. Su costo es el contrario al de la mediana: **infla** las correlaciones con sus propios predictores (Sunshine–Humidity3pm pasa de −0,629 a −0,739), porque lo imputado es una función lineal de ellos. Y como es una imputación determinística, sigue subestimando la variabilidad (3,27 < 3,78).

**Estrategia elegida: C (imputación por regresión).**

**Justificación:** es la única de las tres que conserva todos los casos y todas las ciudades, y distorsiona las correlaciones con el objetivo menos que la mediana. Se reportan sus dos defectos (varianza subestimada y correlaciones internas infladas) y en la Parte 3 se verifica si cambian las conclusiones.

# PARTE 2: Regresión Logística

## 2.1 Preparación de Datos

**Variable objetivo:** `y` (1 si llueve al día siguiente, `RainTomorrow = Yes`). **Predictores candidatos:** las 17 variables numéricas (16 continuas y `RainToday`).

Usamos el dataset con la estrategia elegida (imputación por regresión) y la **misma partición 70/30 estratificada por `RainTomorrow`** para las tres estrategias, de modo que después se puedan comparar. Decisiones que vienen de S1 y S2:

- **Outliers:** S1 mostró colas muy largas (lluvia hasta 371 mm, evaporación hasta 145 mm) y S2 recomendó no depender de los extremos. Las variables con |asimetría| > 1 se **recortan (winsorización) a los percentiles 1 y 99 de la muestra de entrenamiento**.
- **Estandarización:** las variables continuas se estandarizan con la media y desviación del entrenamiento, así los coeficientes son comparables y los odds ratios se leen por **+1 desviación estándar**. `RainToday` queda en 0/1 (el odds ratio es entre "llovió hoy" y "no llovió").
- **Categóricas:** `RainToday` se codifica 0/1. Las direcciones del viento y la ciudad no se incluyen: ninguna fue evaluada en S1 ni validada en S2 (ver limitaciones al final).
- **Partición:** 70/30 estratificada, semilla 42.

In [ ]:
idx_train, idx_test = train_test_split(datos.index, test_size=0.30, stratify=datos['y'], random_state=SEED)
idx_train = np.sort(idx_train); idx_test = np.sort(idx_test)
ytr = datos.loc[idx_train, 'y'].to_numpy(); yte = datos.loc[idx_test, 'y'].to_numpy()
print(f"Entrenamiento: {len(idx_train):,} (lluvia {ytr.mean():.4f}) | Prueba: {len(idx_test):,} (lluvia {yte.mean():.4f}) | Total: {len(datos):,}")
print(f"Un clasificador que siempre dice 'no llueve' acierta {1 - yte.mean():.4f} en prueba (línea base de exactitud).")

class Prep:
    """Winsorización (percentiles 1-99 del entrenamiento) y estandarización de continuas. RainToday queda 0/1."""
    def __init__(self, train, variables):
        self.vars = list(variables)
        self.cont = [v for v in self.vars if v != 'RainToday']
        asim = train[self.cont].skew()
        self.wins = [v for v in self.cont if abs(asim[v]) > 1]
        self.lim = {v: (train[v].quantile(0.01), train[v].quantile(0.99)) for v in self.wins}
        w = self._wins(train)
        self.mu = w[self.cont].mean(); self.sd = w[self.cont].std()
        self.n_recortados = {v: int(((train[v] < self.lim[v][0]) | (train[v] > self.lim[v][1])).sum()) for v in self.wins}

    def _wins(self, d):
        d = d[self.vars].copy()
        for v in self.wins:
            d[v] = d[v].clip(*self.lim[v])
        return d

    def transform(self, d, estandarizar=True):
        d = self._wins(d)
        if estandarizar:
            d[self.cont] = (d[self.cont] - self.mu) / self.sd
        return d

prep = Prep(ds_reg.loc[idx_train], VARS_NUM)
print("Variables recortadas (|asimetría| > 1) y filas recortadas en entrenamiento:", prep.n_recortados)
print("Límites p1-p99:", {v: (round(a, 2), round(b, 2)) for v, (a, b) in prep.lim.items()})
Ztr_all = prep.transform(ds_reg.loc[idx_train]); Zte_all = prep.transform(ds_reg.loc[idx_test])

## 2.2 Selección de Variables

### Colinealidad previa (VIF)

S2 advirtió que hay variables casi repetidas (presión 9 am y 3 pm; ráfaga y viento 3 pm; temperaturas). Calculamos el VIF con todas las variables y eliminamos de a una la de VIF > 10; entre las que superan el umbral se elimina **la que tiene menor |r| con `RainTomorrow`** (así se conserva la más informativa de cada grupo repetido).

In [ ]:
def tabla_vif(Z):
    """VIF de cada variable = diagonal de la inversa de la matriz de correlaciones
    (equivale a regresar cada variable sobre las demás, pero sin ajustar una regresión por variable)."""
    R = np.corrcoef(Z.to_numpy(float), rowvar=False)
    return pd.Series(np.diag(np.linalg.inv(R)), index=Z.columns)

# verificación contra statsmodels en una variable
_v = ['Humidity3pm', 'Sunshine', 'Pressure9am', 'Temp3pm', 'MaxTemp', 'Cloud3pm']
_X = sm.add_constant(Ztr_all[_v], has_constant='add').to_numpy()
assert abs(tabla_vif(Ztr_all[_v])['Temp3pm'] - variance_inflation_factor(_X, _v.index('Temp3pm') + 1)) < 1e-8
vif_inicial = tabla_vif(Ztr_all)
r_y = Ztr_all.corrwith(pd.Series(ytr, index=Ztr_all.index))
vars_vif = list(VARS_NUM); eliminadas = []
while True:
    v = tabla_vif(Ztr_all[vars_vif])
    alto = v[v > 10]
    if alto.empty:
        break
    quitar = min(alto.index, key=lambda a: abs(r_y[a]))
    eliminadas.append([quitar, v[quitar], r_y[quitar]]); vars_vif.remove(quitar)
vif_final = tabla_vif(Ztr_all[vars_vif])

print("Tabla 10. VIF antes y después de depurar (entrenamiento)")
display(pd.DataFrame({'VIF inicial': vif_inicial, 'VIF final': vif_final, 'r con y': r_y}).round(2))
print("Variables eliminadas, en orden:")
display(pd.DataFrame(eliminadas, columns=['Variable', 'VIF al eliminarla', 'r con RainTomorrow']).round(3))
VARS_MOD = vars_vif
PRES = 'Pressure9am' if 'Pressure9am' in VARS_MOD else 'Pressure3pm'
print(f"\nVariables candidatas para los modelos ({len(VARS_MOD)}): {VARS_MOD}")

**Lectura.** Con las 17 variables, los VIF más altos son MaxTemp (23,3), Temp3pm (23,2), Temp9am (18,6), Pressure9am (17,4) y Pressure3pm (17,1). Las presiones superan 17, como advirtió el profesor. Se eliminaron Temp9am, MaxTemp y Pressure3pm y se conservaron Temp3pm y Pressure9am, que tienen mayor |r| con la lluvia de mañana. Con las 14 variables restantes el VIF máximo es 6,4 (Temp3pm), bajo el umbral de 10.

### Tres modelos de regresión logística con ponderación de clase

Como solo ~22 % de los días llueve al día siguiente, ajustamos todos los modelos con **pesos de clase balanceados**: $w_c = n / (2\,n_c)$, que da un peso cercano a 2,2 a los días con lluvia y a 0,64 a los demás. Eso sube el recall a costa de algo de exactitud. Para mantener el control del cálculo, el ajuste se hace con una implementación propia de mínimos cuadrados ponderados iterativos (IRLS), que **verificamos contra `statsmodels`** más abajo.

**Los tres modelos** (la selección se hace con la verosimilitud sin pesos, que es la que define AIC y BIC; el modelo final se ajusta con pesos):
1. **Validado (S1 + S2):** humedad 3 pm, nubosidad 3 pm, horas de sol, lluvia de hoy y presión; son las variables que S2 recomendó por tener relaciones robustas y fuertes.
2. **Stepwise por AIC:** selección bidireccional (entra y sale una variable por paso) partiendo del modelo vacío, entre las variables que pasaron el filtro de VIF.
3. **Reducido:** stepwise por BIC (penaliza más a los modelos grandes) y luego se conservan solo las primeras variables que entraron, hasta el menor número de variables cuyo AUC de entrenamiento queda a menos de 0,003 del modelo BIC completo.

Siguiendo el comentario del profesor a la S2, los modelos son: **M1** guiado por S1 y S2, **M2** por stepwise según AIC y **M3** reducido (selección por BIC, el criterio más estricto, recortada a las variables que más aportan; es la variante AIC/BIC de la plantilla).

In [ ]:
def fit_logit(X, y, w=None, beta0=None, max_iter=60, tol=1e-9):
    """Regresión logística por IRLS (Newton-Raphson) con pesos opcionales. Devuelve coeficientes (con intercepto) y log-verosimilitud ponderada."""
    X = np.column_stack([np.ones(len(X)), np.asarray(X, float)]); y = np.asarray(y, float)
    w = np.ones(len(y)) if w is None else np.asarray(w, float)
    beta = np.zeros(X.shape[1]) if beta0 is None else np.asarray(beta0, float).copy()
    for _ in range(max_iter):
        p = 1 / (1 + np.exp(-(X @ beta)))
        W = w * p * (1 - p)
        paso = np.linalg.solve((X.T * W) @ X, X.T @ (w * (y - p)))
        beta = beta + paso
        if np.abs(paso).max() < tol:
            break
    p = np.clip(1 / (1 + np.exp(-(X @ beta))), 1e-12, 1 - 1e-12)
    return beta, float(np.sum(w * (y * np.log(p) + (1 - y) * np.log(1 - p))))

def cov_robusta(X, y, w, beta):
    """Covarianza sandwich (HC0) del estimador ponderado."""
    X = np.column_stack([np.ones(len(X)), np.asarray(X, float)]); y = np.asarray(y, float)
    p = 1 / (1 + np.exp(-(X @ beta))); W = w * p * (1 - p)
    Hinv = np.linalg.inv((X.T * W) @ X)
    score = X * (w * (y - p))[:, None]
    return Hinv @ (score.T @ score) @ Hinv

def prob(beta, X):
    return 1 / (1 + np.exp(-(np.column_stack([np.ones(len(X)), np.asarray(X, float)]) @ beta)))

def pesos_clase(y):
    y = np.asarray(y); n = len(y)
    return np.where(y == 1, n / (2 * y.sum()), n / (2 * (n - y.sum())))

# verificación de la implementación contra statsmodels (sin pesos y con pesos)
v_chk = ['Humidity3pm', 'Cloud3pm', 'Sunshine', 'RainToday', PRES]
b_mio, _ = fit_logit(Ztr_all[v_chk].values, ytr)
b_sm = sm.Logit(ytr, sm.add_constant(Ztr_all[v_chk])).fit(disp=0)
w_chk = pesos_clase(ytr)
b_mio_w, _ = fit_logit(Ztr_all[v_chk].values, ytr, w_chk)
b_sm_w = sm.GLM(ytr, sm.add_constant(Ztr_all[v_chk]), family=sm.families.Binomial(), freq_weights=w_chk).fit()
print("Diferencia máxima de coeficientes vs. statsmodels (sin pesos):", np.abs(b_mio - b_sm.params.values).max())
print("Diferencia máxima de coeficientes vs. statsmodels (con pesos): ", np.abs(b_mio_w - b_sm_w.params.values).max())
assert np.abs(b_mio - b_sm.params.values).max() < 1e-6 and np.abs(b_mio_w - b_sm_w.params.values).max() < 1e-6
print("Pesos de clase:", {"sin lluvia": round(w_chk[ytr == 0][0], 3), "lluvia": round(w_chk[ytr == 1][0], 3)})

In [ ]:
_cache_ll = {}   # los pasos del stepwise por AIC y por BIC repiten muchos ajustes: se calculan una sola vez

def criterio(Z, y, vs, tipo):
    clave = (id(Z), tuple(sorted(vs)))
    if clave not in _cache_ll:
        _, _cache_ll[clave] = fit_logit(Z[vs].values if vs else np.empty((len(y), 0)), y, tol=1e-7)
    k = len(vs) + 1
    return -2 * _cache_ll[clave] + (2 * k if tipo == 'aic' else k * np.log(len(y)))

def stepwise(Z, y, candidatos, tipo='aic'):
    """Selección bidireccional por AIC o BIC. Devuelve variables finales, orden de entrada y la trayectoria."""
    sel, orden, trayecto = [], [], []
    actual = criterio(Z, y, sel, tipo)
    trayecto.append(['inicio', '(modelo vacío)', actual])
    while True:
        mejor = None
        for v in candidatos:
            prueba = [x for x in sel if x != v] if v in sel else sel + [v]
            c = criterio(Z, y, prueba, tipo)
            if c < actual - 1e-6 and (mejor is None or c < mejor[1]):
                mejor = (v, c)
        if mejor is None:
            break
        v, actual = mejor
        if v in sel:
            sel.remove(v); orden.remove(v); trayecto.append(['sale', v, actual])
        else:
            sel.append(v); orden.append(v); trayecto.append(['entra', v, actual])
    return sel, orden, pd.DataFrame(trayecto, columns=['Paso', 'Variable', tipo.upper()])

Ztr = Ztr_all[VARS_MOD]; Zte = Zte_all[VARS_MOD]
vars_aic, orden_aic, trayecto_aic = stepwise(Ztr, ytr, VARS_MOD, 'aic')
vars_bic, orden_bic, trayecto_bic = stepwise(Ztr, ytr, VARS_MOD, 'bic')
print("Stepwise AIC - trayectoria"); display(trayecto_aic)
print("Variables que el AIC deja fuera:", [v for v in VARS_MOD if v not in vars_aic])
print("Variables que el BIC deja fuera:", [v for v in VARS_MOD if v not in vars_bic])

# modelo reducido: prefijo del orden de entrada del BIC
def auc_train(vs):
    b, _ = fit_logit(Ztr[vs].values, ytr)
    return roc_auc_score(ytr, prob(b, Ztr[vs].values))
auc_bic_total = auc_train(vars_bic)
for k in range(1, len(orden_bic) + 1):
    if auc_bic_total - auc_train(orden_bic[:k]) < 0.003:
        break
vars_red = orden_bic[:k]
print(f"\nReducido: {k} variables {vars_red} (AUC entrenamiento {auc_train(vars_red):.4f} vs. {auc_bic_total:.4f} del BIC completo)")

VARS_M1 = ['Humidity3pm', 'Cloud3pm', 'Sunshine', 'RainToday', PRES]
assert all(v in VARS_MOD for v in VARS_M1)

In [ ]:
w_tr = pesos_clase(ytr)

def metricas(y, p, umbral=0.5):
    pred = (p >= umbral).astype(int)
    return {'accuracy': accuracy_score(y, pred), 'precision': precision_score(y, pred, zero_division=0),
            'recall': recall_score(y, pred), 'f1': f1_score(y, pred), 'auc': roc_auc_score(y, p)}

def construir(nombre, vs):
    beta, _ = fit_logit(Ztr[vs].values, ytr, w_tr)                 # modelo final: con pesos
    p_tr = prob(beta, Ztr[vs].values); p_te = prob(beta, Zte[vs].values)
    return {'nombre': nombre, 'vars': vs, 'beta': beta, 'p_tr': p_tr, 'p_te': p_te,
            'tr': metricas(ytr, p_tr), 'te': metricas(yte, p_te),
            'aic': criterio(Ztr, ytr, vs, 'aic'), 'bic': criterio(Ztr, ytr, vs, 'bic')}

modelos = {
    'M1 Validado S1+S2': construir('M1 Validado S1+S2', VARS_M1),
    'M2 Stepwise AIC': construir('M2 Stepwise AIC', vars_aic),
    'M3 Reducido': construir('M3 Reducido', vars_red),
}

In [ ]:
# variables y odds ratios de cada modelo
todas = [v for v in VARS_MOD]
coef_t = pd.DataFrame({nom: pd.Series(m['beta'][1:], index=m['vars']) for nom, m in modelos.items()}).reindex(todas).dropna(how='all')
or_t = np.exp(coef_t)
print("Tabla 12. Coeficientes (logit, por +1 DE; RainToday por unidad) de los tres modelos"); display(coef_t.round(3))
print("Tabla 13. Odds ratios (exp del coeficiente). OR > 1 sube la chance de lluvia mañana"); display(or_t.round(3))

**Lectura.**

- **Selección.** El stepwise por AIC deja fuera solo Humidity9am y Cloud9am (su información ya está en Humidity3pm y Cloud3pm). El BIC elige las mismas 12 variables: con tantos datos AIC y BIC coinciden. El modelo reducido necesita 8 variables para quedar a menos de 0,003 de AUC del BIC completo.
- **Odds ratios (por +1 DE).** En M2 la humedad a las 3 pm es el efecto más fuerte (OR 2,51), seguida de la ráfaga de viento (2,01; variable que S2 no había recomendado y que el stepwise incorpora segunda), la lluvia de hoy (1,37 si llovió) y la nubosidad (1,31). Las horas de sol (0,59) y la presión (0,69) reducen la chance de lluvia. Las variables del Modelo 1 conservan el signo en los tres modelos.
- **Cuidado con Temp3pm.** Su correlación simple con la lluvia de mañana es negativa (−0,19) pero su coeficiente en M2 es positivo (OR 1,59). Es un efecto condicional a la humedad y al sol; no debe leerse como "más calor, más lluvia".

## 2.3 Bootstrap del Modelo - Usando la técnica de la S2

Aplicamos bootstrap no paramétrico (igual que en S2) al modelo final: se remuestrean con reemplazo las filas de entrenamiento, se reajusta el modelo ponderado en cada remuestra y se toman los percentiles 2,5 y 97,5 de cada coeficiente. Los comparamos con el **intervalo tradicional** (Wald con errores robustos, que es el que corresponde a un estimador ponderado). Un parámetro es **inestable** si su IC bootstrap cruza el 0 o si su signo cambia en más del 1 % de las remuestras.

Como S2 mostró que los días de una misma ciudad no son independientes (el intervalo de la proporción se ensanchó ~8 veces al remuestrear ciudades), repetimos el bootstrap **remuestreando ciudades completas**.

In [ ]:
# el bootstrap se aplica al mejor modelo; la regla se fijó de antemano y la comparación completa está en 2.5
# regla de selección: mayor AUC en prueba; si hay empate práctico (<= 0,002) se prefiere el modelo con menos variables
auc_max = max(m['te']['auc'] for m in modelos.values())
elegibles = [k for k, m in modelos.items() if auc_max - m['te']['auc'] <= 0.002]
FINAL = min(elegibles, key=lambda k: len(modelos[k]['vars']))
VARS_FINAL = modelos[FINAL]['vars']
print(f"Elegibles (AUC a menos de 0,002 del mejor): {elegibles}")
print(f"MODELO FINAL: {FINAL} con {len(VARS_FINAL)} variables: {VARS_FINAL}")

In [ ]:
rng_seed = SEED
Xf_tr = Ztr[VARS_FINAL].values
beta_f = modelos[FINAL]['beta']
B_FILAS, B_CIUDADES = 1000, 500
grupos_tr = datos.loc[idx_train, 'Location'].to_numpy()

def _ajustar_lote(X, y, w, lote, beta0):
    """Reajusta el modelo para un lote de remuestras. Cada remuestra se representa con el número de veces
    que sale cada fila (pesos de multiplicidad), que da exactamente los mismos coeficientes que duplicar filas."""
    return np.array([fit_logit(X, y, w * c, beta0=beta0, tol=1e-7)[0] for c in lote])

def bootstrap_logit(X, y, w, B, seed, beta0, grupos=None, tam_lote=25, n_jobs=-1):
    """Bootstrap de los coeficientes. Las remuestras se sortean en orden con una sola semilla (resultados
    reproducibles) y se ajustan en paralelo en todos los núcleos disponibles (n_jobs=-1)."""
    rng = np.random.default_rng(seed)
    n = len(y)
    if grupos is not None:
        ciudades, cod = np.unique(grupos, return_inverse=True)

    def remuestra():
        if grupos is None:
            return np.bincount(rng.integers(0, n, n), minlength=n).astype(np.uint8)
        elegidas = rng.choice(len(ciudades), size=len(ciudades), replace=True)
        return np.bincount(elegidas, minlength=len(ciudades))[cod].astype(np.uint8)

    def lotes():
        for ini in range(0, B, tam_lote):
            yield [remuestra() for _ in range(min(tam_lote, B - ini))]

    res = Parallel(n_jobs=n_jobs)(delayed(_ajustar_lote)(X, y, w, lote, beta0) for lote in lotes())
    return np.vstack(res)

import time
t0 = time.time()
boot_filas = bootstrap_logit(Xf_tr, ytr, w_tr, B_FILAS, SEED, beta_f)
print(f"Bootstrap por filas: {B_FILAS} remuestras en {time.time() - t0:.0f} s")
t0 = time.time()
boot_ciud = bootstrap_logit(Xf_tr, ytr, w_tr, B_CIUDADES, SEED, beta_f, grupos=grupos_tr)
print(f"Bootstrap por ciudades: {B_CIUDADES} remuestras en {time.time() - t0:.0f} s")

# intervalo tradicional: Wald con errores robustos (sandwich)
cov = cov_robusta(Xf_tr, ytr, w_tr, beta_f)
se = np.sqrt(np.diag(cov))
nombres = ['Intercepto'] + VARS_FINAL
z975 = stats.norm.ppf(0.975)

def resumen_boot(boot):
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    signo = np.mean(np.sign(boot) == np.sign(beta_f), axis=0)
    return lo, hi, signo

lo_f, hi_f, sg_f = resumen_boot(boot_filas)
lo_c, hi_c, sg_c = resumen_boot(boot_ciud)
tabla_boot = pd.DataFrame({
    'Coef.': beta_f, 'OR': np.exp(beta_f),
    'IC tradicional (coef.)': [f"[{b - z975 * s:.3f}; {b + z975 * s:.3f}]" for b, s in zip(beta_f, se)],
    'IC tradicional (OR)': [f"[{np.exp(b - z975 * s):.3f}; {np.exp(b + z975 * s):.3f}]" for b, s in zip(beta_f, se)],
    'IC bootstrap filas (coef.)': [f"[{a:.3f}; {b:.3f}]" for a, b in zip(lo_f, hi_f)],
    'OR bootstrap filas': [f"[{np.exp(a):.3f}; {np.exp(b):.3f}]" for a, b in zip(lo_f, hi_f)],
    'Signo conservado (filas)': sg_f,
    'IC bootstrap ciudades (coef.)': [f"[{a:.3f}; {b:.3f}]" for a, b in zip(lo_c, hi_c)],
    'Signo conservado (ciudades)': sg_c,
}, index=nombres)
inestable_f = [(n, ) for n, a, b, s in zip(nombres, lo_f, hi_f, sg_f) if n != 'Intercepto' and (a < 0 < b or s < 0.99)]
inestable_c = [(n, ) for n, a, b, s in zip(nombres, lo_c, hi_c, sg_c) if n != 'Intercepto' and (a < 0 < b or s < 0.99)]
print("Tabla 14a. Coeficientes del modelo final: IC tradicional y bootstrap (95 %)")
display(tabla_boot[['Coef.', 'IC tradicional (coef.)', 'IC bootstrap filas (coef.)', 'Signo conservado (filas)',
                    'IC bootstrap ciudades (coef.)', 'Signo conservado (ciudades)']])
print("Tabla 14b. Odds ratios del modelo final: IC tradicional y bootstrap por filas (95 %)")
display(tabla_boot[['OR', 'IC tradicional (OR)', 'OR bootstrap filas']])
print("Parámetros inestables (bootstrap por filas):  ", [x[0] for x in inestable_f] or 'ninguno')
print("Parámetros inestables (bootstrap por ciudades):", [x[0] for x in inestable_c] or 'ninguno')
ancho = pd.DataFrame({'Ancho IC tradicional': 2 * z975 * se, 'Ancho IC bootstrap filas': hi_f - lo_f,
                      'Ancho IC bootstrap ciudades': hi_c - lo_c}, index=nombres)
ancho['Razón ciudades / tradicional'] = ancho['Ancho IC bootstrap ciudades'] / ancho['Ancho IC tradicional']
print("Anchos de los intervalos"); display(ancho.round(3))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 5.2), sharey=True)
y_pos = np.arange(len(VARS_FINAL))[::-1]
for ax, esc, titulo in [(ejes[0], 'coef', 'Coeficiente (logit)'), (ejes[1], 'or', 'Odds ratio')]:
    f = (lambda v: v) if esc == 'coef' else np.exp
    for k, (lo, hi, col, lab, off) in enumerate([
            (beta_f[1:] - z975 * se[1:], beta_f[1:] + z975 * se[1:], OCHRE, 'Tradicional (robusto)', 0.22),
            (lo_f[1:], hi_f[1:], TEAL, 'Bootstrap filas', 0),
            (lo_c[1:], hi_c[1:], TERRA, 'Bootstrap ciudades', -0.22)]):
        ax.hlines(y_pos + off, f(lo), f(hi), color=col, lw=2.2, label=lab)
    ax.plot(f(beta_f[1:]), y_pos, 'o', color=INK, ms=4, zorder=3)
    ax.axvline(0 if esc == 'coef' else 1, color=STEM, lw=1, ls='--')
    ax.set_xlabel(titulo); ax.set_yticks(y_pos); ax.set_yticklabels(VARS_FINAL)
ejes[0].legend(frameon=False, fontsize=8, loc='lower right')
fig.suptitle('Figura 6 - Intervalos al 95 % del modelo final: tradicional, bootstrap por filas y bootstrap por ciudades', x=0.01, ha='left')
fig.tight_layout(); guardar(fig, 's3_fig_bootstrap_coeficientes.png'); plt.show()

**Lectura.**

- **Bootstrap por filas.** El IC bootstrap coincide con el tradicional en los 13 parámetros (anchos casi idénticos, por ejemplo Humidity3pm 0,071 contra 0,074). Ninguno cruza el 0 y el signo se conserva en el 100 % de las 1.000 remuestras: no hay parámetros inestables. Es el mismo resultado de la S2 para las medias y correlaciones, y se debe a que con unos 100.000 días el error de muestreo es muy pequeño.
- **Bootstrap por ciudades.** Al remuestrear ciudades completas, los intervalos son entre 1,6 y 4,7 veces más anchos. Es la misma lección de la S2 (allí el intervalo de la proporción de lluvia se ensanchó unas 8 veces): los días de una misma ciudad no son independientes. Los más afectados son WindGustSpeed (4,7 veces), Pressure9am (4,4), WindSpeed3pm (4,3) y Humidity3pm (3,9). Aun así, 11 de los 12 coeficientes conservan su signo y su IC no cruza el 0.
- **MinTemp es el único parámetro inestable.** Su IC por ciudades [−0,281; 0,032] cruza el 0 y su signo se conserva en 95,8 % de las remuestras. Tiene un efecto pequeño (OR 0,89) que depende de qué ciudades entren, así que conviene interpretarlo con cautela.

## 2.4 Diagnóstico de Supuestos

Los supuestos clásicos de la regresión logística se definen sobre la verosimilitud **sin pesos**, y la ponderación de clase solo cambia el intercepto y algo la calibración. Por eso el diagnóstico se hace sobre el modelo final con las mismas variables, ajustado sin pesos. Se revisan:

1. **Multicolinealidad:** VIF de las variables finales.
2. **Observaciones influyentes:** distancia de Cook para regresión logística, $D_i = \dfrac{r_i^2\,h_i}{k\,(1-h_i)^2}$ (con $r_i$ el residuo de Pearson y $h_i$ el apalancamiento). Se reporta el máximo y cuántas superan la regla $4/n$ (que con ~100.000 filas es muy sensible), y se reajusta el modelo sin el **1 % más influyente** para ver si cambian los coeficientes.
3. **Linealidad en el logit:** se agrega el término cuadrático de cada variable continua y se compara con una prueba de razón de verosimilitud. Con 100.000 filas casi cualquier curvatura es "significativa", así que se mira también la mejora del AUC y el gráfico del logit empírico por deciles.
4. **Residuos:** residuos de devianza y gráfico de residuos agrupados.

In [ ]:
# 1. VIF final
vif_mod = tabla_vif(Ztr[VARS_FINAL])
print("Tabla 15. VIF del modelo final"); display(vif_mod.to_frame('VIF').round(2).T)

# 2. observaciones influyentes (modelo sin pesos)
b_u, ll_u = fit_logit(Xf_tr, ytr)
Xc = np.column_stack([np.ones(len(Xf_tr)), Xf_tr])
p_u = 1 / (1 + np.exp(-(Xc @ b_u))); W_u = p_u * (1 - p_u)
Hinv = np.linalg.inv((Xc.T * W_u) @ Xc)
h_ii = W_u * np.einsum('ij,jk,ik->i', Xc, Hinv, Xc)
r_pear = (ytr - p_u) / np.sqrt(W_u)
cook = (r_pear ** 2 * h_ii) / (Xc.shape[1] * (1 - h_ii) ** 2)
umbral_cook = 4 / len(ytr)
corte_1pct = np.quantile(cook, 0.99)
infl = cook >= corte_1pct
print(f"Distancia de Cook: máximo {cook.max():.5f} (referencia habitual de preocupación: D > 1)")
print(f"Regla 4/n ({umbral_cook:.2e}): marca {(cook > umbral_cook).sum():,} observaciones ({(cook > umbral_cook).mean() * 100:.1f} %), demasiado sensible con n grande.")
print(f"Se reajusta sin el 1 % más influyente: {infl.sum():,} observaciones con D >= {corte_1pct:.2e}")
print(f"Apalancamiento máximo: {h_ii.max():.4f} (promedio esperado {Xc.shape[1] / len(ytr):.5f})")

b_sin, _ = fit_logit(Xf_tr[~infl], ytr[~infl])
cambio = pd.DataFrame({'Coef. completo': b_u, 'Coef. sin influyentes': b_sin}, index=nombres)
cambio['Cambio relativo (%)'] = (cambio.iloc[:, 1] / cambio.iloc[:, 0] - 1) * 100
auc_con = roc_auc_score(yte, prob(b_u, Zte[VARS_FINAL].values)); auc_sin = roc_auc_score(yte, prob(b_sin, Zte[VARS_FINAL].values))
print("Tabla 16. Efecto de quitar el 1 % de observaciones más influyentes (modelo sin pesos)")
display(cambio.round(3))
print(f"AUC en prueba: {auc_con:.4f} con todas vs. {auc_sin:.4f} sin el 1 % más influyente")

top = np.argsort(cook)[::-1][:5]
print("Las 5 más influyentes (en las variables originales):")
display(datos.loc[idx_train[top], ['Location', 'Date', 'y'] + VARS_FINAL].assign(Date=lambda d: d['Date'].dt.date, Cook=cook[top]))

**Lectura.**

- **Multicolinealidad.** El VIF máximo del modelo final es 6,2 (Temp3pm); todos están bajo 10.
- **Influencia.** La distancia de Cook máxima es 0,00045, muy lejos de 1: ninguna observación individual mueve el modelo. La regla 4/n marcaría 7,9 % de las filas, pero con ~100.000 filas es demasiado sensible. Al quitar el 1 % más influyente (996 filas), los coeficientes grandes cambian poco (Humidity3pm +11 %, WindGustSpeed +14 %, Sunshine +1 %), ningún signo cambia y el AUC en prueba queda igual (0,8597 contra 0,8596). Los que más cambian son los de efecto pequeño: Rainfall (+81 %) y Evaporation (+65 %). Las filas más influyentes son días de lluvia muy fuerte hoy (38 a 56 mm), en 4 de 5 casos seguidos de un día sin lluvia.

In [ ]:
# 3. linealidad en el logit: término cuadrático + prueba de razón de verosimilitud
continuas = [v for v in VARS_FINAL if v != 'RainToday']
filas = []
for v in continuas:
    j = VARS_FINAL.index(v)
    Xq = np.column_stack([Xf_tr, Ztr[v].values ** 2])
    bq, llq = fit_logit(Xq, ytr)
    lr = 2 * (llq - ll_u); pval = stats.chi2.sf(lr, 1)
    p_q = prob(bq, np.column_stack([Zte[VARS_FINAL].values, Zte[v].values ** 2]))
    auc_q = roc_auc_score(yte, p_q)
    filas.append([v, bq[-1], lr, pval, auc_q - auc_con])
lin = pd.DataFrame(filas, columns=['Variable', 'Coef. del término cuadrático', 'LR (1 gl)', 'p', 'Δ AUC en prueba']).set_index('Variable')
print("Tabla 17. Prueba de curvatura (término cuadrático)"); display(lin.round(4))

# logit empírico por deciles
orig = prep.transform(ds_reg.loc[idx_train], estandarizar=False)
n_pan = len(continuas); ncol = 5; nfil = int(np.ceil(n_pan / ncol))
fig, ejes = plt.subplots(nfil, ncol, figsize=(14, 2.7 * nfil))
for ax, v in zip(ejes.ravel(), continuas):
    q = pd.qcut(orig[v], 10, duplicates='drop')
    g = pd.DataFrame({'x': orig[v].to_numpy(), 'y': ytr}).groupby(q.to_numpy()).agg(x=('x', 'mean'), p=('y', 'mean'))
    g['logit'] = np.log(g['p'] / (1 - g['p']))
    ax.plot(g['x'], g['logit'], 'o-', color=TEAL, ms=4); ax.set_title(v, fontsize=9); ax.tick_params(labelsize=7)
for ax in ejes.ravel()[n_pan:]:
    ax.axis('off')
fig.suptitle('Figura 7 - Logit empírico por deciles de cada variable continua (linealidad en el logit)', x=0.01, ha='left')
fig.tight_layout(); guardar(fig, 's3_fig_linealidad_logit.png'); plt.show()

**Lectura.** Con 100.000 filas la prueba de razón de verosimilitud detecta curvatura (p < 0,05) en 9 de las 11 variables continuas, pero la mejora de AUC al agregar el término cuadrático es de 0,0006 o menos en todas (Tabla 17): la curvatura no tiene valor predictivo. En el logit empírico (Figura 7) las relaciones de humedad, sol, presión, nubosidad, temperatura y ráfaga son monótonas y casi rectas. Las que se apartan de una recta son Rainfall (sube rápido entre 0 y 3 mm y luego se aplana), MinTemp (sube rápido bajo ~8 °C, queda plana hasta ~17 °C y vuelve a subir) y Evaporation (plana hasta unos 3 mm y luego baja). Se mantiene la forma lineal por parsimonia; una transformación o spline para Rainfall y MinTemp queda como mejora posible.

In [ ]:
# 4. residuos
dev = np.sign(ytr - p_u) * np.sqrt(-2 * (ytr * np.log(np.clip(p_u, 1e-12, 1)) + (1 - ytr) * np.log(np.clip(1 - p_u, 1e-12, 1))))
print(f"Residuos de devianza: media {dev.mean():.3f}, |dev| > 2 en {np.mean(np.abs(dev) > 2) * 100:.1f} % de las filas, máximo {np.abs(dev).max():.2f}")
_, ll0 = fit_logit(np.empty((len(ytr), 0)), ytr)
print(f"Devianza: {-2 * ll0:,.0f} (modelo nulo) vs. {-2 * ll_u:,.0f} (modelo final), una reducción de {(1 - ll_u / ll0) * 100:.1f} %")

dfb = pd.DataFrame({'p': p_u, 'y': ytr})
dfb['bin'] = pd.qcut(dfb['p'], 50, duplicates='drop')
gb = dfb.groupby('bin', observed=True).agg(p=('p', 'mean'), obs=('y', 'mean'), n=('y', 'size'))
gb['res'] = gb['obs'] - gb['p']; gb['se'] = 2 * np.sqrt(gb['p'] * (1 - gb['p']) / gb['n'])
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.3))
ejes[0].plot(gb['p'], gb['res'], 'o', color=TEAL, ms=4)
ejes[0].plot(gb['p'], gb['se'], color=OCHRE, lw=1); ejes[0].plot(gb['p'], -gb['se'], color=OCHRE, lw=1)
ejes[0].axhline(0, color=TERRA, lw=1)
ejes[0].set_xlabel('Probabilidad predicha (promedio del grupo)'); ejes[0].set_ylabel('Residuo promedio (observado - predicho)')
ejes[0].set_title('Residuos agrupados (50 grupos, bandas ±2 EE)')
ejes[1].plot(gb['p'], gb['obs'], 'o', color=TEAL, ms=4); ejes[1].plot([0, 1], [0, 1], color=TERRA, lw=1)
ejes[1].set_xlabel('Probabilidad predicha'); ejes[1].set_ylabel('Frecuencia observada'); ejes[1].set_title('Calibración (modelo sin pesos)')
fig.suptitle('Figura 8 - Residuos y calibración del modelo final', x=0.01, ha='left')
fig.tight_layout(); guardar(fig, 's3_fig_residuos_calibracion.png'); plt.show()
print(f"Grupos con residuo fuera de ±2 EE: {(np.abs(gb['res']) > gb['se']).sum()} de {len(gb)}")

**Lectura.** Los residuos de devianza no muestran valores extremos (máximo 3,14; solo 3,2 % supera 2 en valor absoluto), y la devianza baja de 105.935 (modelo nulo) a 73.298 (−30,8 %). La calibración sigue la diagonal (Figura 8): 7 de los 50 grupos quedan fuera de ±2 errores estándar (por azar se esperarían 2 o 3), con desviaciones de hasta 2,5 puntos de probabilidad, sobre todo subestimación entre 0,1 y 0,3 y sobreestimación entre 0,5 y 0,7. Es una calibración razonable.

Ojo: esto vale para la versión sin pesos. El modelo ponderado entrega probabilidades más altas a propósito, así que su umbral 0,5 no equivale a una probabilidad real de 50 %.

## 2.5 Evaluación de Desempeño

Matrices de confusión (entrenamiento y prueba) y curvas ROC de los tres modelos.

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(11, 6.8))
for j, (nom, m) in enumerate(modelos.items()):
    for i, (yy, pp, lab) in enumerate([(ytr, m['p_tr'], 'Entrenamiento'), (yte, m['p_te'], 'Prueba')]):
        cm = confusion_matrix(yy, (pp >= 0.5).astype(int))
        ax = ejes[i, j]; ax.imshow(cm, cmap='YlGnBu'); ax.grid(False)
        for a in range(2):
            for b in range(2):
                ax.text(b, a, f"{cm[a, b]:,}", ha='center', va='center', color='white' if cm[a, b] > cm.max() / 2 else INK)
        ax.set_xticks([0, 1]); ax.set_xticklabels(['No llueve', 'Llueve']); ax.set_yticks([0, 1]); ax.set_yticklabels(['No llueve', 'Llueve'])
        ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title(f"{nom} - {lab}", fontsize=10)
fig.suptitle('Figura 4 - Matrices de confusión (umbral 0,5)', x=0.01, ha='left')
fig.tight_layout(); guardar(fig, 's3_fig_matrices_confusion.png'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 5))
for (nom, m), col in zip(modelos.items(), [OCHRE, TEAL, TERRA]):
    fpr, tpr, _ = roc_curve(yte, m['p_te'])
    ax.plot(fpr, tpr, color=col, lw=1.8, label=f"{nom} (AUC {m['te']['auc']:.3f})")
ax.plot([0, 1], [0, 1], color=STEM, lw=1, ls='--')
ax.set_xlabel('Tasa de falsos positivos'); ax.set_ylabel('Recall (tasa de verdaderos positivos)')
ax.set_title('Figura 5 - Curvas ROC en prueba'); ax.legend(frameon=False, loc='lower right', fontsize=8)
fig.tight_layout(); guardar(fig, 's3_fig_roc.png'); plt.show()

### Comparación de Modelos

In [ ]:
filas = []
for nom, m in modelos.items():
    filas.append([nom, len(m['vars']), m['aic'], m['bic'],
                  *[m['tr'][k] for k in ('accuracy', 'precision', 'recall', 'f1', 'auc')],
                  *[m['te'][k] for k in ('accuracy', 'precision', 'recall', 'f1', 'auc')]])
cols = pd.MultiIndex.from_tuples([('', 'Modelo'), ('', 'Variables'), ('Criterios', 'AIC'), ('Criterios', 'BIC')] +
                                 [('Entrenamiento', k) for k in ('Exactitud', 'Precisión', 'Recall', 'F1', 'AUC')] +
                                 [('Prueba', k) for k in ('Exactitud', 'Precisión', 'Recall', 'F1', 'AUC')])
comp = pd.DataFrame(filas, columns=cols).set_index(('', 'Modelo')); comp.index.name = 'Modelo'
print("Tabla 11. Desempeño de los tres modelos (umbral 0,5; modelos ajustados con pesos de clase)")
display(comp.round(4))

**Lectura.**

- **Discriminación.** AUC en prueba: 0,844 (M1), 0,860 (M2) y 0,858 (M3). La brecha entre M2 y M3 es de 0,002 y entre M2 y M1 de 0,016. No hay sobreajuste: M2 tiene AUC 0,858 en entrenamiento y 0,860 en prueba.
- **Efecto de la ponderación.** Sube el recall a 0,756–0,763 (se detectan unos 3 de cada 4 días de lluvia). El costo es la precisión, 0,49–0,51: aproximadamente la mitad de las alertas de lluvia son falsas. La exactitud en prueba es 0,766 (M1), 0,783 (M2) y 0,781 (M3), contra 0,776 de decir siempre "no llueve"; M1 queda incluso por debajo de esa línea de base. Con pesos de clase el objetivo deja de ser la exactitud y pasa a ser no dejar pasar días de lluvia.

**Modelo seleccionado: M2 (stepwise por AIC).**

**Justificación:** Las curvas ROC de M2 y M3 casi se superponen y M1 queda debajo (Figura 5). El **modelo final es M2 (stepwise por AIC)**: tiene el mayor AUC en prueba (0,860) y M3 queda a 0,0022, justo fuera de la zona de empate de 0,002 definida de antemano. M2 también tiene el menor AIC (73.324 contra 73.616 de M3 y 76.301 de M1). Como esta regla usa el conjunto de prueba, la elección es levemente optimista; la diferencia M2–M3 es tan pequeña que cualquiera de los dos se podría defender, y M3 es la alternativa si se prefiere un modelo más simple.

---
# PARTE 3: Análisis Comparativo de Imputación

Cerramos el círculo: el **mismo modelo final** (mismas variables, mismos pesos de clase) se ajusta con los tres datasets del Paso 3 usando **la misma partición** de filas (en el dataset por eliminación solo quedan las filas completas de esa partición). Cada dataset tiene su propia winsorización y estandarización calculadas con su entrenamiento.

Como los conjuntos de prueba no son los mismos (eliminación solo tiene casos completos, de 26 ciudades), se reporta el desempeño en **su propia prueba** y en una **prueba común**: las filas completas de la partición de prueba, que son iguales en las tres estrategias (no tienen ningún valor imputado).

In [ ]:
te_comun = ds_elim.index.intersection(idx_test)
resultados, coefs_ds = {}, {}
for nombre, ds in [('Eliminación', ds_elim), ('Mediana', ds_med), ('Regresión', ds_reg)]:
    tr_i = ds.index.intersection(idx_train); te_i = ds.index.intersection(idx_test)
    pr = Prep(ds.loc[tr_i], VARS_MOD)
    Xa = pr.transform(ds.loc[tr_i])[VARS_FINAL].values; ya = ds.loc[tr_i, 'y'].to_numpy(); wa = pesos_clase(ya)
    b, _ = fit_logit(Xa, ya, wa)
    se_ = np.sqrt(np.diag(cov_robusta(Xa, ya, wa, b)))
    p_prop = prob(b, pr.transform(ds.loc[te_i])[VARS_FINAL].values)
    p_com = prob(b, pr.transform(ds_elim.loc[te_comun])[VARS_FINAL].values)
    m_prop = metricas(ds.loc[te_i, 'y'].to_numpy(), p_prop); m_com = metricas(ds_elim.loc[te_comun, 'y'].to_numpy(), p_com)
    resultados[nombre] = {'n_tr': len(tr_i), 'n_te': len(te_i), 'prop': m_prop, 'com': m_com, 'ancho': float(np.mean(2 * z975 * se_[1:]))}
    coefs_ds[nombre] = (b, se_)

filas = []
for nom, r in resultados.items():
    filas.append([nom, r['n_tr'], r['n_te'], r['prop']['auc'], r['prop']['accuracy'], r['prop']['recall'],
                  len(te_comun), r['com']['auc'], r['com']['accuracy'], r['com']['recall'], r['ancho']])
cols = pd.MultiIndex.from_tuples([('', 'Estrategia'), ('Muestra', 'n entrenamiento'), ('Muestra', 'n prueba propia'),
                                  ('Prueba propia', 'AUC'), ('Prueba propia', 'Exactitud'), ('Prueba propia', 'Recall'),
                                  ('Prueba común', 'n'), ('Prueba común', 'AUC'), ('Prueba común', 'Exactitud'), ('Prueba común', 'Recall'), ('Incertidumbre', 'Ancho IC prom. (coef.)')])
impacto = pd.DataFrame(filas, columns=cols).set_index(('', 'Estrategia')); impacto.index.name = 'Estrategia'
print("Tabla 18. Desempeño del modelo final según la estrategia de imputación")
display(impacto.round(4))

In [ ]:
tabla_coef = {}
for nom, (b, s) in coefs_ds.items():
    for i, v in enumerate(nombres):
        tabla_coef[(v, nom)] = (b[i], b[i] - z975 * s[i], b[i] + z975 * s[i])
filas = []
for v in nombres:
    fila = [v]
    for nom in coefs_ds:
        c, lo, hi = tabla_coef[(v, nom)]
        fila += [c, f"[{lo:.3f}; {hi:.3f}]"]
    filas.append(fila)
cols = ['Variable'] + [f"{k} {t}" for k in coefs_ds for t in ('coef.', 'IC 95 %')]
coef_cmp = pd.DataFrame(filas, columns=cols).set_index('Variable')
print("Tabla 19. Coeficientes (por +1 DE) e IC robustos del modelo final con cada dataset"); display(coef_cmp)

ref = np.array([tabla_coef[(v, 'Regresión')][0] for v in nombres])
variacion = pd.DataFrame({k: [(tabla_coef[(v, k)][0] / r - 1) * 100 for v, r in zip(nombres, ref)] for k in ('Eliminación', 'Mediana')}, index=nombres)
variacion.columns = ['Eliminación vs. Regresión (%)', 'Mediana vs. Regresión (%)']
cambia_signo = [v for v in VARS_FINAL if len({np.sign(tabla_coef[(v, k)][0]) for k in coefs_ds}) > 1]
print("Variación relativa del coeficiente respecto a la imputación por regresión"); display(variacion.round(1))
print("Variables cuyo signo cambia según la estrategia:", cambia_signo or 'ninguna')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.6))
y_pos = np.arange(len(VARS_FINAL))[::-1]
for (nom, col), off in zip([('Eliminación', TERRA), ('Mediana', OCHRE), ('Regresión', TEAL)], (-0.22, 0, 0.22)):
    c = np.array([tabla_coef[(v, nom)][0] for v in VARS_FINAL]); lo = np.array([tabla_coef[(v, nom)][1] for v in VARS_FINAL]); hi = np.array([tabla_coef[(v, nom)][2] for v in VARS_FINAL])
    ax.hlines(y_pos + off, lo, hi, color=col, lw=2.2, label=nom); ax.plot(c, y_pos + off, 'o', color=col, ms=4)
ax.axvline(0, color=STEM, lw=1, ls='--'); ax.set_yticks(y_pos); ax.set_yticklabels(VARS_FINAL)
ax.set_xlabel('Coeficiente por +1 DE (logit), IC 95 %'); ax.legend(frameon=False, fontsize=8)
ax.set_title('Figura 9 - Coeficientes del modelo final según la estrategia de imputación')
fig.tight_layout(); guardar(fig, 's3_fig_impacto_imputacion.png'); plt.show()

**Lectura.**

- **Predicción.** En la prueba común (17.471 días completos, sin ningún valor imputado) el AUC es 0,881 (eliminación), 0,878 (mediana) y 0,880 (regresión): diferencias de 0,003 o menos, con exactitud entre 0,793 y 0,803 y recall entre 0,783 y 0,794. En la prueba propia de cada estrategia, el AUC de eliminación (0,881) supera al de las otras dos (0,858 y 0,860); esa brecha viene de la población (las 26 ciudades completas son más fáciles de predecir) y no de la imputación, porque desaparece en la prueba común.
- **Coeficientes.** Ninguna variable cambia de signo y todos los IC excluyen el 0 en las tres estrategias, pero las magnitudes sí varían. La mediana **atenúa** Sunshine (−0,35 contra −0,53 con regresión, −35 %) y Evaporation (−0,05 contra −0,14, −62 %), porque reemplaza casi la mitad de sus valores por una constante. La regresión baja Humidity3pm a 0,92 (contra 1,20 con mediana y 1,22 con eliminación), porque las variables imputadas llevan información de la humedad y comparten su efecto. En eliminación MinTemp (−0,39 contra −0,12) y Temp3pm (0,68 contra 0,46) son más fuertes, pero ahí se mezcla el cambio de población con el de imputación.

### Conclusión sobre Imputación

**¿La estrategia de imputación cambió las conclusiones?** No en lo esencial. En la prueba común el AUC es 0,881 (A), 0,878 (B) y 0,880 (C); ningún coeficiente cambia de signo y todos los IC excluyen el 0 en las tres estrategias. Lo que cambia es la **magnitud** de algunos efectos: la mediana (B) atenúa Sunshine (−35 %) y Evaporation (−62 %) respecto de la regresión (C), y C baja Humidity3pm cerca de 24 % respecto de A y B (0,92 contra 1,22 y 1,20). Además, los IC de A son 1,7 a 1,8 veces más anchos (ancho promedio 0,115 contra 0,063 de B y 0,068 de C) porque quedan 40.618 días de entrenamiento en vez de 99.535.

**¿Qué estrategia recomiendas?** **C, imputación por regresión.** Conserva los 142.193 casos y las 49 ciudades (A solo 40,9 % de las filas y 26 ciudades), no atenúa los efectos como B y recupera mejor las correlaciones con el objetivo. Si el único fin fuera predecir, cualquiera de las tres sirve, pero A deja el modelo limitado a las ciudades que miden todo.

**Limitaciones:**
- Los coeficientes de A no son comparables del todo con los de B y C: cambian la población (26 ciudades) y la imputación a la vez.
- La imputación determinística subestima la variabilidad de lo imputado y, por lo tanto, los IC de las variables imputadas son algo optimistas; la imputación múltiple lo corregiría.
- La prueba común tiene solo 17.471 días de 26 ciudades, las más fáciles de predecir, así que favorece a A.
- Entre 60 % y 77 % de lo imputado cae en ciudades que nunca miden la variable y ahí no se puede validar.

In [ ]:
# resultados para el informe
comp.to_csv(PROCESADOS / 's3_desempeno_modelos.csv', encoding='utf-8')
tabla_boot.to_csv(PROCESADOS / 's3_bootstrap_coeficientes.csv', encoding='utf-8')
impacto.to_csv(PROCESADOS / 's3_impacto_imputacion.csv', encoding='utf-8')
tabla_imp.to_csv(PROCESADOS / 's3_imputacion_regresion.csv', encoding='utf-8')
print("Tablas guardadas en data/processed/ (s3_*.csv)")

---
# PARTE 4: Conclusiones Integradas

## Resumen S1 → S2 → S3

**Hallazgos S1 (Exploración):** el objetivo está desbalanceado (llueve al día siguiente en 22,4 % de los días). 21 de las 23 variables tienen faltantes y Sunshine, Evaporation y la nubosidad llegan a entre 38 % y 48 %, con estaciones que no las miden. La humedad a las 3 pm es la variable más asociada con la lluvia de mañana (r = 0,446), seguida de la nubosidad, las horas de sol y haber llovido hoy. Presión 9 am y 3 pm (0,96) y las temperaturas están casi repetidas. La lluvia tiene una cola muy larga (hasta 371 mm) y los días de una misma ciudad no son independientes.

**Hallazgos S2 (Validación):** los intervalos clásicos de la S1 son confiables (percentil, BCa y clásico coinciden) y las pruebas de humedad y nubosidad se confirman por permutación y se cumplen en las 49 ciudades. Las correlaciones evaluadas son robustas (la de la temperatura de las 9 am es débil) y la persistencia de la lluvia (46,4 % contra 15,2 %) se reproduce con Monte Carlo. La media de la lluvia es sensible a los extremos, el ~40 % de faltantes de la nubosidad no es al azar y el intervalo de la proporción de lluvia es unas 8 veces más ancho al remuestrear ciudades.

**Hallazgos S3 (Modelamiento):** la imputación por regresión conserva los 142.193 casos y las 49 ciudades y distorsiona las correlaciones con el objetivo menos que la mediana. El modelo final (stepwise por AIC, 12 variables) alcanza AUC 0,860 y recall 0,763 en prueba, contra 0,844 y 0,756 del modelo guiado solo por S1 y S2. Los coeficientes son estables al remuestrear filas; al remuestrear ciudades solo MinTemp pierde estabilidad. La estrategia de imputación no cambia la capacidad predictiva ni el signo de los efectos, pero sí la magnitud de Sunshine y Evaporation.

| Tema | S1 encontró | S2 validó | S3 hizo |
|---|---|---|---|
| Faltantes | 21 de 23 variables con faltantes; Sunshine, Evaporation y nubosidad entre 38 % y 48 % | El ~40 % de la nubosidad no es al azar: hay estaciones que no la miden | Imputación por regresión con predictores de la matriz de S1; supuesto MAR declarado como limitación; comparación de 3 estrategias |
| Predictores | Correlaciones: humedad 3 pm (0,45), nubosidad, sol, lluvia de hoy | IC bootstrap estrechos, ninguna correlación cruza 0; presiones casi repetidas (0,96) | Predictores de la imputación, Modelo 1 y depuración por VIF (presiones sobre 17) |
| Desbalance | Llueve en 22,4 % de los días | Cualquier modelo debe superar 77,6 % de exactitud | Pesos de clase (2,23 para lluvia); evaluación con recall y AUC |
| Valores extremos | Lluvia con cola muy larga (máximo 371 mm) | La media de lluvia es sensible; usar mediana o "llovió / no llovió" | Winsorización p1–p99 y `RainToday` como predictor |
| Dependencia entre días | Días seguidos de 49 ciudades | Remuestrear ciudades ensancha el IC unas 8 veces | Bootstrap por ciudades: IC 1,6 a 4,7 veces más anchos; MinTemp resulta inestable |

## Modelo Final

**Variables incluidas (12):** Humidity3pm, WindGustSpeed, Sunshine, Pressure9am, RainToday, Temp3pm, WindSpeed3pm, Cloud3pm, WindSpeed9am, Evaporation, Rainfall y MinTemp. Quedan fuera Humidity9am y Cloud9am, cuya información ya está en Humidity3pm y Cloud3pm.

**Desempeño en prueba (42.658 días):** AUC 0,860; exactitud 0,783; precisión 0,511; recall 0,763; F1 0,612 (umbral 0,5, con pesos de clase). La línea base de "siempre no llueve" acierta 0,776.

**Interpretación de Odds Ratios (por +1 desviación estándar; RainToday por unidad):**
- Humidity3pm 2,51 y WindGustSpeed 2,01: son los efectos más fuertes y suben la chance de lluvia al día siguiente.
- Cloud3pm 1,31 y RainToday 1,37: más nubosidad y haber llovido hoy suben la chance.
- Sunshine 0,59 y Pressure9am 0,69: más sol y más presión la bajan.
- Temp3pm 1,59 es un efecto condicional (con humedad y sol fijos); su correlación simple con la lluvia de mañana es negativa y no debe leerse como "más calor, más lluvia".
- Evaporation 0,87, WindSpeed3pm 0,84, WindSpeed9am 0,89 y MinTemp 0,89 bajan levemente la chance; Rainfall 1,08 la sube levemente. MinTemp es el único parámetro inestable al remuestrear ciudades.

## Recomendaciones para Toma de Decisiones

1. **Usar el modelo como alerta de "plan B", no como pronóstico exacto.** En la prueba, de 9.563 días con lluvia el modelo avisó 7.292 (76 %), pero de las 14.285 alertas solo 51 % fueron lluvia real. Conviene cuando dejar pasar un día de lluvia es más caro que una falsa alarma (cosecha, obra o evento al aire libre).
2. **Vigilar las señales que más pesan:** la humedad a las 3 pm, la ráfaga de viento, las horas de sol y la presión, además de si hoy llovió.
3. **Combinar con la simulación de la S2.** La lluvia persiste (46,4 % si hoy llovió, 15,2 % si no) y planificar con días independientes subestima el riesgo de rachas de varios días: el modelo diario y la simulación mensual se complementan.
4. **No aplicar el modelo sin calibrar a una ciudad concreta.** La incertidumbre real es de 1,6 a 4,7 veces la del intervalo clásico y la lluvia varía mucho entre ciudades (S2). Para una ciudad conviene ajustar el umbral o incorporar la ciudad al modelo.
5. **Ajustar el umbral al costo de decidir mal.** El 0,5 no se optimizó: bajarlo sube el recall y las falsas alarmas; subirlo hace lo contrario.
6. **Leer con cautela las magnitudes de Sunshine, Evaporation y las nubosidades,** porque son las más imputadas y las más sensibles a la estrategia de faltantes.

## Limitaciones del Análisis

- El supuesto MAR es parcial: entre 60 % y 77 % de lo imputado cae en ciudades que nunca miden la variable y no se puede validar ahí.
- La imputación determinística subestima la variabilidad e infla las correlaciones con sus predictores; una imputación múltiple lo corregiría.
- La imputación se ajustó con todas las filas (entrenamiento y prueba), aunque nunca con el objetivo.
- La ciudad y la dirección del viento no entran al modelo; S2 recomendó considerar la ciudad, y el bootstrap por ciudades solo mide la dependencia, no la corrige.
- El umbral de 0,5 no se optimizó y la precisión es de 0,51.
- La validación es aleatoria; no se probó separando por tiempo.

---
## Checklist de Integración

- [x] ¿Referencio matriz de correlaciones de S1? (sección 0 y selección de predictores de la imputación)
- [x] ¿Uso correlaciones estables de S2? (Modelo 1, depuración por VIF y sección 0)
- [x] ¿Trato outliers de S1? (winsorización p1–p99 en 2.1)
- [x] ¿Aplico bootstrap de S2 al modelo? (2.3, por filas y por ciudades)
- [x] ¿Construyo IC bootstrap para OR? (Tabla 14b y Figura 6)
- [x] ¿Mi análisis muestra progresión S1→S2→S3? (Parte 4)

---
*Notebook completado por Jeremmy Díaz Alfaro, Karim Zaid Sánchez y Laura Arenas Kosin - 27 de octubre, 2026*